# 🖋️ Impala67 — Lokales Handschrift-Modell Training (Digital Ink)
Dieses Notebook trainiert ein kompaktes CRNN (1D-CNN + BiLSTM + CTC) für Vektor-Striche.
Das fertige Modell ist nur **ca. 3 MB** groß und läuft über `onnxruntime-web` direkt im Browser.

### Schritte:
1. **GPU aktivieren:** *Laufzeit -> Laufzeittyp ändern -> T4-GPU*
2. **(Optional):** Deine in Impala67 exportierte Datei `my_handwriting_samples.json` in Colab hochladen.
3. Alle Zellen ausführen (*Laufzeit -> Alle ausführen*).
4. Am Ende wird `model.onnx` automatisch heruntergeladen.

In [ ]:
# 1. Setup & Pakete
!pip install -q onnx onnxruntime onnxscript
import math, random, json, os, time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | Gerät: {device}")

In [ ]:
# 2. Vokabular (86 Zeichen, CTC-Blank = 0)
CHARS = [
    "<blank>",
    "a", "b", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l", "m",
    "n", "o", "p", "q", "r", "s", "t", "u", "v", "w", "x", "y", "z",
    "A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M",
    "N", "O", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z",
    "ä", "ö", "ü", "Ä", "Ö", "Ü", "ß",
    "0", "1", "2", "3", "4", "5", "6", "7", "8", "9",
    " ",
    ".", ",", "!", "?", "-", "+", ":", "/", "*", "=", "(", ")", "@", "#", "%"
]
CHAR_TO_IDX = {c: i for i, c in enumerate(CHARS)}
IDX_TO_CHAR = {i: c for i, c in enumerate(CHARS)}
VOCAB_SIZE = len(CHARS)
print(f"Vokabular: {VOCAB_SIZE} Zeichen")

In [ ]:
# 3. Synthetischer Strichgenerator (Online Handwriting Primitives)
GLYPH_STROKES = {
    "a": [[(0.8, 0.3), (0.4, 0.2), (0.1, 0.5), (0.3, 0.9), (0.7, 0.8), (0.8, 0.3)], [(0.8, 0.2), (0.8, 0.9)]],
    "b": [[(0.2, 0.0), (0.2, 0.9)], [(0.2, 0.4), (0.7, 0.3), (0.8, 0.7), (0.5, 0.9), (0.2, 0.8)]],
    "c": [[(0.8, 0.3), (0.4, 0.2), (0.1, 0.5), (0.4, 0.9), (0.8, 0.8)]],
    "d": [[(0.8, 0.4), (0.4, 0.3), (0.2, 0.6), (0.5, 0.9), (0.8, 0.8)], [(0.8, 0.0), (0.8, 0.9)]],
    "e": [[(0.1, 0.6), (0.8, 0.5), (0.6, 0.2), (0.2, 0.4), (0.2, 0.8), (0.7, 0.9)]],
    "f": [[(0.7, 0.1), (0.4, 0.0), (0.3, 0.4), (0.3, 0.9)], [(0.1, 0.4), (0.6, 0.4)]],
    "g": [[(0.8, 0.3), (0.4, 0.2), (0.1, 0.5), (0.4, 0.9), (0.8, 0.8), (0.8, 0.3)], [(0.8, 0.3), (0.8, 1.2), (0.3, 1.3)]],
    "h": [[(0.2, 0.0), (0.2, 0.9)], [(0.2, 0.4), (0.5, 0.3), (0.8, 0.5), (0.8, 0.9)]],
    "i": [[(0.4, 0.3), (0.4, 0.9)], [(0.4, 0.1), (0.42, 0.12)]],
    "j": [[(0.5, 0.3), (0.5, 1.2), (0.2, 1.3)], [(0.5, 0.1), (0.52, 0.12)]],
    "k": [[(0.2, 0.0), (0.2, 0.9)], [(0.7, 0.3), (0.2, 0.6)], [(0.3, 0.5), (0.8, 0.9)]],
    "l": [[(0.3, 0.0), (0.3, 0.9)]],
    "m": [[(0.1, 0.3), (0.1, 0.9)], [(0.1, 0.4), (0.4, 0.3), (0.5, 0.9)], [(0.5, 0.4), (0.8, 0.3), (0.9, 0.9)]],
    "n": [[(0.2, 0.3), (0.2, 0.9)], [(0.2, 0.4), (0.5, 0.3), (0.8, 0.5), (0.8, 0.9)]],
    "o": [[(0.5, 0.2), (0.2, 0.4), (0.2, 0.7), (0.5, 0.9), (0.8, 0.7), (0.8, 0.4), (0.5, 0.2)]],
    "p": [[(0.2, 0.3), (0.2, 1.3)], [(0.2, 0.4), (0.7, 0.3), (0.8, 0.7), (0.5, 0.9), (0.2, 0.8)]],
    "q": [[(0.8, 0.4), (0.4, 0.3), (0.2, 0.6), (0.5, 0.9), (0.8, 0.8)], [(0.8, 0.3), (0.8, 1.3)]],
    "r": [[(0.2, 0.3), (0.2, 0.9)], [(0.2, 0.5), (0.5, 0.3), (0.8, 0.4)]],
    "s": [[(0.8, 0.3), (0.4, 0.2), (0.2, 0.4), (0.6, 0.6), (0.8, 0.8), (0.4, 0.9), (0.1, 0.8)]],
    "t": [[(0.4, 0.1), (0.4, 0.9)], [(0.2, 0.3), (0.7, 0.3)]],
    "u": [[(0.2, 0.3), (0.2, 0.8), (0.5, 0.9), (0.8, 0.8)], [(0.8, 0.3), (0.8, 0.9)]],
    "v": [[(0.1, 0.3), (0.5, 0.9)], [(0.5, 0.9), (0.9, 0.3)]],
    "w": [[(0.1, 0.3), (0.3, 0.9)], [(0.3, 0.9), (0.5, 0.4)], [(0.5, 0.4), (0.7, 0.9)], [(0.7, 0.9), (0.9, 0.3)]],
    "x": [[(0.2, 0.3), (0.8, 0.9)], [(0.8, 0.3), (0.2, 0.9)]],
    "y": [[(0.2, 0.3), (0.5, 0.8)], [(0.8, 0.3), (0.2, 1.3)]],
    "z": [[(0.2, 0.3), (0.8, 0.3), (0.2, 0.9), (0.8, 0.9)]],
    "ä": [[(0.8, 0.3), (0.4, 0.2), (0.1, 0.5), (0.3, 0.9), (0.7, 0.8), (0.8, 0.3)], [(0.8, 0.2), (0.8, 0.9)], [(0.3, 0.05), (0.32, 0.07)], [(0.7, 0.05), (0.72, 0.07)]],
    "ö": [[(0.5, 0.2), (0.2, 0.4), (0.2, 0.7), (0.5, 0.9), (0.8, 0.7), (0.8, 0.4), (0.5, 0.2)], [(0.35, 0.05), (0.37, 0.07)], [(0.65, 0.05), (0.67, 0.07)]],
    "ü": [[(0.2, 0.3), (0.2, 0.8), (0.5, 0.9), (0.8, 0.8)], [(0.8, 0.3), (0.8, 0.9)], [(0.35, 0.05), (0.37, 0.07)], [(0.65, 0.05), (0.67, 0.07)]],
    "ß": [[(0.2, 0.9), (0.2, 0.0)], [(0.2, 0.0), (0.6, 0.1), (0.6, 0.4), (0.2, 0.4)], [(0.2, 0.4), (0.7, 0.5), (0.6, 0.8), (0.2, 0.8)]],
    "0": [[(0.5, 0.0), (0.1, 0.4), (0.1, 0.6), (0.5, 1.0), (0.9, 0.6), (0.9, 0.4), (0.5, 0.0)]],
    "1": [[(0.2, 0.2), (0.5, 0.0), (0.5, 1.0)]],
    "2": [[(0.2, 0.2), (0.5, 0.0), (0.8, 0.2), (0.2, 0.9), (0.8, 0.9)]],
    "3": [[(0.2, 0.1), (0.8, 0.1), (0.5, 0.4), (0.8, 0.7), (0.4, 1.0), (0.2, 0.9)]],
    "4": [[(0.7, 0.1), (0.2, 0.6), (0.9, 0.6)], [(0.7, 0.1), (0.7, 1.0)]],
    "5": [[(0.8, 0.0), (0.2, 0.0), (0.2, 0.4), (0.7, 0.5), (0.7, 0.8), (0.3, 1.0)]],
    "6": [[(0.7, 0.1), (0.3, 0.4), (0.2, 0.7), (0.5, 1.0), (0.8, 0.8), (0.8, 0.6), (0.3, 0.6)]],
    "7": [[(0.1, 0.0), (0.8, 0.0), (0.4, 1.0)]],
    "8": [[(0.5, 0.4), (0.3, 0.2), (0.5, 0.0), (0.7, 0.2), (0.5, 0.4), (0.2, 0.7), (0.5, 1.0), (0.8, 0.7), (0.5, 0.4)]],
    "9": [[(0.5, 0.5), (0.2, 0.3), (0.5, 0.0), (0.8, 0.3), (0.8, 0.7), (0.5, 1.0)]],
    " ": [],
    ".": [[(0.4, 0.9), (0.45, 0.92)]],
    ",": [[(0.4, 0.85), (0.35, 1.0)]],
    "!": [[(0.5, 0.0), (0.5, 0.7)], [(0.5, 0.9), (0.52, 0.92)]],
    "?": [[(0.2, 0.2), (0.5, 0.0), (0.8, 0.2), (0.5, 0.5), (0.5, 0.7)], [(0.5, 0.9), (0.52, 0.92)]],
    "-": [[(0.2, 0.5), (0.8, 0.5)]],
    "+": [[(0.5, 0.2), (0.5, 0.8)], [(0.2, 0.5), (0.8, 0.5)]],
    "=": [[(0.2, 0.4), (0.8, 0.4)], [(0.2, 0.6), (0.8, 0.6)]],
}
for k in list(GLYPH_STROKES.keys()):
    if k.islower():
        GLYPH_STROKES[k.upper()] = [[(p[0] * 1.1, p[1] * 1.1) for p in s] for s in GLYPH_STROKES[k]]

WORDS = ['Notiz', 'Aufgabe', 'Projekt', 'Idee', 'Treffen', 'Wichtig', 'Datum', 'Ziel', 'Haus', 'Buch', 'Schule', 'Arbeit', 'Code', 'Test', 'Fehler', 'Version', 'heute', 'morgen', 'gut', 'sehr', 'machen', 'sehen', 'haben', 'sein', 'und', 'oder', 'aber', 'wenn', 'dann', 'nicht', 'ein', 'eine', 'das', 'der', 'die', 'Plan', 'Zeile', 'Stift', 'Farbe', 'Text', 'Karte', 'Wort', 'Seite', '123', '456', '789', '2026', '100%', 'Nr.1']

def interpolate(p0, p1, step=0.05):
    dx, dy = p1[0] - p0[0], p1[1] - p0[1]
    dist = math.hypot(dx, dy)
    if dist < step: return [p1]
    n = max(1, int(dist / step))
    return [(p0[0] + dx * (i / n), p0[1] + dy * (i / n)) for i in range(1, n + 1)]

def word_to_strokes(word, step=0.05, jitter=0.015):
    strokes = []
    cx = 0.0
    slant = random.uniform(-0.15, 0.2)
    scale_y = random.uniform(0.85, 1.15)
    for char in word:
        glyph = GLYPH_STROKES.get(char, GLYPH_STROKES.get('c'))
        cw = 0.6 if char != ' ' else 0.4
        for raw in glyph:
            if not raw: continue
            pts = []
            for i, p in enumerate(raw):
                x = cx + p[0] * cw + p[1] * slant + random.gauss(0, jitter)
                y = p[1] * scale_y + random.gauss(0, jitter)
                if i == 0: pts.append((x, y))
                else: pts.extend(interpolate(pts[-1], (x, y), step))
            if pts: strokes.append(pts)
        cx += cw + random.uniform(0.05, 0.15)
    return strokes

def strokes_to_feats(strokes):
    all_pts = [p for s in strokes for p in s]
    if not all_pts: return []
    h = max(0.1, max(p[1] for p in all_pts) - min(p[1] for p in all_pts))
    scale = 1.0 / h
    feats = []
    lx, ly = None, None
    for s in strokes:
        if not s: continue
        if lx is not None: feats.append(((s[0][0] - lx) * scale, (s[0][1] - ly) * scale, 0.0))
        lx, ly = s[0][0], s[0][1]
        for p in s[1:]:
            feats.append(((p[0] - lx) * scale, (p[1] - ly) * scale, 1.0))
            lx, ly = p[0], p[1]
    return feats

In [ ]:
# 4. Nutzerdaten laden (falls my_handwriting_samples.json vorhanden)
user_samples = []
if os.path.exists('my_handwriting_samples.json'):
    with open('my_handwriting_samples.json', 'r', encoding='utf-8') as f:
        for item in json.load(f):
            txt = str(item.get('text', '')).strip()
            st = [[(float(p[0]), float(p[1])) for p in s.get('pts', [])] for s in item.get('strokes', [])]
            if txt and st:
                f_pts = strokes_to_feats(st)
                if len(f_pts) >= 5: user_samples.append((f_pts, txt))
    print(f"⭐ {len(user_samples)} eigene Handschrift-Samples geladen!")
else:
    print("ℹ️ Keine my_handwriting_samples.json gefunden — Training läuft rein auf synthetischen Daten.")

In [ ]:
# 5. Dataset & DataLoader
class InkDataset(Dataset):
    def __init__(self, size=2400):
        self.size = size
    def __len__(self): return self.size
    def __getitem__(self, idx):
        if user_samples and random.random() < 0.4:
            feats, word = random.choice(user_samples)
        else:
            word = random.choice(WORDS)
            feats = strokes_to_feats(word_to_strokes(word))
        return torch.tensor(feats, dtype=torch.float32), torch.tensor([CHAR_TO_IDX.get(c, 0) for c in word], dtype=torch.long), word

def pad_collate(batch):
    feats, tgts, words = zip(*batch)
    in_lens = torch.tensor([f.shape[0] for f in feats], dtype=torch.long)
    tgt_lens = torch.tensor([t.shape[0] for t in tgts], dtype=torch.long)
    max_len = max(in_lens)
    padded = torch.zeros(len(feats), max_len, 3, dtype=torch.float32)
    for i, f in enumerate(feats): padded[i, :f.shape[0], :] = f
    return padded, torch.cat(tgts), in_lens, tgt_lens, words

train_loader = DataLoader(InkDataset(size=2400), batch_size=32, shuffle=True, collate_fn=pad_collate)

In [ ]:
# 6. Modell-Architektur (1D-CNN + BiLSTM + CTC)
class HandwritingCRNN(nn.Module):
    def __init__(self, in_features=3, hidden_size=128, vocab_size=VOCAB_SIZE):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_features, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
        )
        self.lstm = nn.LSTM(input_size=128, hidden_size=hidden_size, num_layers=2, bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size * 2, vocab_size)
        self.log_softmax = nn.LogSoftmax(dim=-1)

    def forward(self, x):
        feats = self.cnn(x.transpose(1, 2)).permute(2, 0, 1)
        lstm_out, _ = self.lstm(feats)
        return self.log_softmax(self.fc(lstm_out))

model = HandwritingCRNN().to(device)
print(f"Modell erstellt. Parameter: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 7. Training
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
EPOCHS = 20

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for inputs, targets, in_lens, tgt_lens, words in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        log_probs = model(inputs)
        out_lens = torch.clamp(in_lens // 2, min=1).to(device)
        loss = criterion(log_probs, targets, out_lens, tgt_lens)
        if torch.isnan(loss) or torch.isinf(loss): continue
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoche {epoch:02d}/{EPOCHS} | Loss: {total_loss / len(train_loader):.4f}")

In [ ]:
# 8. Export nach model.onnx & Download
model.eval()
dummy = torch.randn(1, 120, 3, dtype=torch.float32)
torch.onnx.export(
    model.cpu(), dummy, 'model.onnx', opset_version=14,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch', 1: 'seq_len'}, 'output': {0: 'seq_out', 1: 'batch'}},
    dynamo=False
)
print(f"Fertig! model.onnx ist {os.path.getsize('model.onnx') / 1024 / 1024:.2f} MB groß.")
try:
    from google.colab import files
    files.download('model.onnx')
except Exception as e:
    print("Download manuell über den Dateibrowser links starten.")